<a href="https://colab.research.google.com/github/m-av-i/Fundamentos/blob/main/Notebooks/coleta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print ("hello rafa")

hello rafa


In [6]:
import httpx
from bs4 import BeautifulSoup
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
}

paginas = [
    "https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa?b_start:int=0",
    "https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa?b_start:int=30"
]

def acessar_pagina(link):
    try:
        response = httpx.get(link, headers=headers, timeout=20.0, follow_redirects=True)
        bs = BeautifulSoup(response.text, "html.parser")


        titulos = bs.select("#content-core .titulo")

        if not titulos:
            print(f"Nenhuma notícia encontrada em {link}. O layout pode ter mudado novamente.")
            return

        for h2_titulo in titulos:
            container = h2_titulo.find_parent("li")
            if not container:
                container = h2_titulo.find_parent("div") # Fallback

            titulo_txt = h2_titulo.text.strip()
            link_tag = h2_titulo.find("a")
            link_noticia = link_tag["href"] if link_tag else None

            if not link_noticia: continue


            span_data = container.find("span", class_="date")
            raw_date = span_data.text.strip() if span_data else "Data não encontrada"

            parts = raw_date.split()
            data = parts[0] if len(parts) > 0 else "-"
            horario = parts[1] if len(parts) > 1 else "-"

            numero_nota = titulo_txt.split("nº")[-1].strip() if "nº" in titulo_txt else "N/A"

            paragrafos = "Não extraído"
            try:

                #sleep pra não ser kickado rs
                time.sleep(0.5)
                res_detalhe = httpx.get(link_noticia, headers=headers, timeout=10.0, follow_redirects=True)
                bs_detalhe = BeautifulSoup(res_detalhe.text, "html.parser")


                corpo = bs_detalhe.find("div", id="parent-fieldname-text")
                if not corpo:
                    corpo = bs_detalhe.find("div", property="rnews:articleBody")

                if corpo:
                    paragrafos = "\n".join([p.text.strip() for p in corpo.find_all("p") if p.text.strip()])
                    paragrafos = paragrafos[:200] + "..." # Truncando pro print não ficar gigante
            except Exception:
                pass

            print(f"Nota: {numero_nota}")
            print(f"Data: {data} | Hora: {horario}")
            print(f"Título: {titulo_txt}")
            print(f"Link: {link_noticia}")
            print(f"Texto: {paragrafos}")
            print("-" * 50)

    except Exception as e:
        print(f"Erro ao processar página {link}: {e}")


for p in paginas:
    print(f">>> Lendo: {p}")
    acessar_pagina(p)

>>> Lendo: https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa?b_start:int=0
Nota: N/A
Data: Data | Hora: não
Título: Escalada de hostilidades no Oriente Médio
Link: https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/escalada-de-hostilidades-no-oriente-medio
Texto: O Governo brasileiro manifesta profunda preocupação com a escalada de hostilidades na região do Golfo, que representa uma grave ameaça à paz e à segurança internacionais, com potenciais impactos human...
--------------------------------------------------
Nota: N/A
Data: Data | Hora: não
Título: Acidente aéreo na Bolívia
Link: https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/acidente-aereo-na-bolivia
Texto: O Governo brasileiro expressa solidariedade ao Governo e ao povo bolivianos pelo acidente envolvendo avião da Força Aérea da Bolívia, em 27/2. Expressa, ainda, condolências às famílias das vítimas e d...
--------------------------------------------------